<a href="https://colab.research.google.com/github/antargfx/antargfx.github.io/blob/main/%F0%9F%94%98Image_Upscale_and_Remove_Duplicates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📖 ব্যবহারবিধি

1️⃣ **Mount Drive** সেল রান করুন এবং Google Drive সংযুক্ত করুন।

2️⃣ Google Drive-এ থাকা **Folder** বা **ZIP File** নির্বাচন করুন।

3️⃣ প্রয়োজন অনুযায়ী সেটিংস নির্বাচন করুন:

* **Outscale**: 2x, 3x বা 4x
* **Face Enhance**: মুখের ছবি হলে ON
* **Output Format**: JPG, PNG বা WEBP
* **Quality**: JPG/WebP-এর জন্য
* **min_file_size_mb**: সর্বনিম্ন ফাইল সাইজ কতো হবে সিলেক্ট করুন
* **max_file_size_mb**: সর্বোচ্চ ফাইল সাইজ কতো হবে সিলেক্ট করুন
* **auto_download_zip**: এটি চালু থাকলে কাজ শেষ হওয়ার পর ZIP ফাইল নিজে থেকেই ডাউনলোড শুরু হবে।

4️⃣ **▶ Process & Upscale** বাটনে ক্লিক করুন।

5️⃣ প্রসেস শেষ হলে আপস্কেল করা ছবি এবং ZIP ফাইল Google Drive-এর:

```text
image upscaled/
```

ফোল্ডারে সংরক্ষিত হবে।

উদাহরণ:

```text
Input Folder: Antar

Output Folder: image upscaled/Antar_upscaled
```

💡 Portrait ছবির জন্য **Face Enhance ON** ব্যবহার করলে সবচেয়ে ভালো ফলাফল পাওয়া যাবে।


In [ ]:
#@title 0️⃣1️⃣Mount Drive

import os

# ============ DRIVE OPTIONS ============
mount_drive = True  #@param {type:"boolean"}

DRIVE_MOUNTED = False
DRIVE_OUTPUT = None

if mount_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive')

        DRIVE_BASE = "/content/drive/MyDrive/image upscaled"
        DRIVE_OUTPUT = DRIVE_BASE

        if not os.path.exists(DRIVE_BASE):
            os.makedirs(DRIVE_BASE)
            print("✅ Created folder: image upscaled/")
        else:
            print("✅ Found folder: image upscaled/")

        DRIVE_MOUNTED = True
        print(f"\n📁 Results will be saved to:")
        print(f"   {DRIVE_OUTPUT}\n")

    except Exception as e:
        print(f"❌ Drive mount failed: {e}")
        print("⚠️ Continuing without Drive - results will be local only\n")
else:
    print("ℹ️ Drive not mounted - results will be local only\n")

# Clone Real-ESRGAN
%cd /content/
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

# Install dependencies
print("⏳ Installing dependencies...")
!pip install -q facexlib
!pip install -q gfpgan
!pip install -q -r requirements.txt
!pip install -q basicsr-fixed
!pip install -e .

# Download ALL models
print("\n📦 Downloading models (this may take 1-2 minutes)...\n")

print("⏳ [1/2] Downloading RealESRGAN_x4plus (official)...")
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P experiments/pretrained_models



print("⏳ [2/2] Downloading AESRGAN (best for faces)...")
!wget -q https://github.com/stroking-fishes-ml-corp/A-ESRGAN/releases/download/v1.0.0/A_ESRGAN_Single.pth -P experiments/pretrained_models

# Verify installation
from IPython.display import clear_output, HTML
clear_output()

print("\033[92m✅ Real-ESRGAN Installation successful\033[0m")
if DRIVE_MOUNTED:
    print(f"\033[92m✅ Drive connected: {DRIVE_OUTPUT}\033[0m\n")
else:
    print(f"\033[93m⚠️  Drive not connected (local only)\033[0m\n")

# Check all models
models = [
    ("RealESRGAN_x4plus", "experiments/pretrained_models/RealESRGAN_x4plus.pth", "Good - Fast, reliable"),
    ("UltraSharp V2", "experiments/pretrained_models/4xUltraSharp.pth", "Better - More detail, same speed"),
    ("AESRGAN", "experiments/pretrained_models/A_ESRGAN_Single.pth", "Best - Faces/portraits, slower")
]

print("📦 Available models:\n")
for name, path, description in models:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"   \033[92m✅ {name:<20} ({size_mb:5.1f}MB)\033[0m - {description}")
    else:
        print(f"   \033[91m❌ MISSING: {name}\033[0m")

# Keep Alive - Using HTML audio (doesn't block cell completion)
display(HTML('''
<audio loop autoplay controls>
  <source src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" type="audio/mp4">
</audio>
'''))

print("\n🎵 Keep-alive enabled (prevents disconnection)")
print("\n" + "="*60)

In [ ]:

#@title 0️⃣2️⃣Select, Upscale & Save
#@markdown ==================================================================
#@markdown ### 🔘 Upscale Guide
#@markdown - **Outscale** = Upscaling factor (2x, 3x, or 4x enlargement)
#@markdown - **facial enhance** = Usually uses a face restoration model in addition to upscaling.
#@markdown - **Output format** = Supported formats: png, jpeg, jpg, webp.
#@markdown - **jpg_quality** = Higher values produce better image quality and larger file sizes.
#@markdown - **min_file_size_mb** = Minimum allowed input file size in megabytes.
#@markdown - **max_file_size_mb** = Maximum allowed input file size in megabytes.
#@markdown - **auto_download_zip** = Auto-download output ZIP if checked.
#@markdown
#@markdown
#@markdown ==================================================================
# ============================================================
# ── MODEL & UPSCALE SETTINGS ────────────────────────────────
# ============================================================
model_choice    = "AESRGAN"
outscale        = 4          #@param {type:"slider", min:2, max:4, step:1}
tile_size       = 512
tile_pad        = 15
face_enhance    = False      #@param {type:"boolean"}
output_format   = "jpg"      #@param ["png", "jpeg", "jpg", "webp"]

# ============================================================
# ── FILE SIZE CONTROL ────────────────────────────────────────
# ============================================================
jpg_quality      = 100  #@param {type:"slider", min:70, max:100, step:5}
min_file_size_mb = 4    #@param {type:"slider", min:0, max:20, step:1}
max_file_size_mb = 40   #@param {type:"slider", min:0, max:100, step:1}

# ============================================================
# ── DOWNLOAD SETTINGS ────────────────────────────────────────
# ============================================================
auto_download_zip = False # @param {"type":"boolean"}


# ============================================================
# IMPORTS
# ============================================================
import os
import re
import time
import shutil
import zipfile
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display, clear_output, HTML
from PIL import Image


# ============================================================
# CONFIG
# ============================================================
DRIVE_ROOT = "/content/drive/MyDrive"

upload_folder = "upload"
result_folder = "results"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff", ".tif"}

MODEL_PATHS = {
    "RealESRGAN_x4plus": "experiments/pretrained_models/RealESRGAN_x4plus.pth",
    "AESRGAN":           "experiments/pretrained_models/A_ESRGAN_Single.pth"
}
selected_model = MODEL_PATHS[model_choice]


# Check Google Drive
DRIVE_MOUNTED = os.path.exists("/content/drive/MyDrive")
DRIVE_OUTPUT  = "/content/drive/MyDrive/image upscaled"

if DRIVE_MOUNTED:
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)


# ============================================================
# HELPERS
# ============================================================
def sanitize_filename(filename):
    safe = re.sub(r"[^\w\.-]", "_", filename)
    if "." not in safe:
        safe += ".jpg"
    return safe


def copy_images_from_folder(folder_path, destination):
    """Recursively copy all images from a folder into destination."""
    copied = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            ext = os.path.splitext(file.lower())[1]
            if ext in IMAGE_EXTENSIONS:
                src      = os.path.join(root, file)
                rel      = os.path.relpath(root, folder_path)
                prefix   = rel.replace(os.sep, "_")
                safe     = sanitize_filename(file)
                if prefix != ".":
                    safe = f"{prefix}_{safe}"
                shutil.copy2(src, os.path.join(destination, safe))
                copied += 1
    return copied


def extract_images_from_zip(zip_path, destination):
    """Extract only image files from a ZIP into destination."""
    extracted = 0
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.filelist:
            if info.is_dir():
                continue
            ext = os.path.splitext(info.filename.lower())[1]
            if ext in IMAGE_EXTENSIONS:
                safe = sanitize_filename(os.path.basename(info.filename))
                out  = os.path.join(destination, safe)
                with open(out, "wb") as f:
                    f.write(zf.read(info.filename))
                extracted += 1
    return extracted


def get_source_basename(path):
    """
    Return a clean base name for the selected source:
      - Folder  → folder name
      - ZIP     → ZIP stem (no extension)
    """
    base = os.path.basename(path.rstrip("/\\"))
    if base.lower().endswith(".zip"):
        base = os.path.splitext(base)[0]
    return re.sub(r"[^\w\.-]", "_", base)


def build_output_name(source_basename, fmt):
    """
    Build the result folder / ZIP name:
      <source_basename>_<YYYYMMDD_HHMMSS>_<ext>
    """
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{source_basename}_{ts}_{fmt}"


def save_with_quality(img, path, fmt, quality):
    if fmt in ("jpg", "jpeg"):
        img.convert("RGB").save(path, "JPEG", quality=quality, optimize=False, subsampling=0)
    elif fmt == "png":
        img.save(path, "PNG", optimize=True, compress_level=0)
    elif fmt == "webp":
        img.save(path, "WEBP", quality=quality, method=0)


def enforce_min_size(output_path, fmt, quality, min_mb):
    """Upscale 25% with Lanczos until file meets minimum size (max 2 passes)."""
    if min_mb <= 0:
        return
    target = min_mb * 1024 * 1024
    resample = getattr(Image, "Resampling", Image).LANCZOS
    for _ in range(2):
        if os.path.getsize(output_path) >= target:
            break
        mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"   🔸 Size {mb:.2f} MB < {min_mb} MB — Lanczos 25% upscale...")
        img = Image.open(output_path)
        w, h = img.size
        img = img.resize((int(w * 1.25), int(h * 1.25)), resample)
        save_with_quality(img, output_path, fmt, quality)


def enforce_max_size(output_path, fmt, quality, max_mb):
    """Reduce quality / size until file fits within maximum size."""
    if max_mb <= 0:
        return
    target = max_mb * 1024 * 1024
    if os.path.getsize(output_path) <= target:
        return
    print(f"   🔹 Output exceeds {max_mb} MB — compressing...")
    img = Image.open(output_path)
    resample = getattr(Image, "Resampling", Image).LANCZOS

    if fmt in ("jpg", "jpeg"):
        q = quality
        while os.path.getsize(output_path) > target and q > 20:
            q -= 5
            img.convert("RGB").save(output_path, "JPEG", quality=q, optimize=True, subsampling=1)
        print(f"   ✅ Reduced to {os.path.getsize(output_path)/(1024*1024):.2f} MB (JPEG q={q})")

    elif fmt == "webp":
        q = quality
        while os.path.getsize(output_path) > target and q > 20:
            q -= 5
            img.save(output_path, "WEBP", quality=q, method=6)
        print(f"   ✅ Reduced to {os.path.getsize(output_path)/(1024*1024):.2f} MB (WebP q={q})")

    elif fmt == "png":
        for _ in range(10):
            if os.path.getsize(output_path) <= target:
                break
            w, h = img.size
            img   = img.resize((int(w * 0.9), int(h * 0.9)), resample)
            img.save(output_path, "PNG", optimize=True, compress_level=9)
        print(f"   ✅ Reduced to {os.path.getsize(output_path)/(1024*1024):.2f} MB")


# ============================================================
# STEP 1 — SCAN GOOGLE DRIVE & BUILD DROPDOWN
# ============================================================
print("🔍 Scanning Google Drive for folders and ZIP files…")

items = []
for root, dirs, files in os.walk(DRIVE_ROOT):
    for d in dirs:
        full = os.path.join(root, d)
        rel  = full.replace(DRIVE_ROOT, "")
        items.append((f"📁 Folder | {rel}", full))
    for f in files:
        if f.lower().endswith(".zip"):
            full = os.path.join(root, f)
            rel  = full.replace(DRIVE_ROOT, "")
            items.append((f"📦 ZIP | {rel}", full))

items.sort(key=lambda x: x[0].lower())
print(f"✅ Found {len(items)} items\n")


# ============================================================
# STEP 2 — UI WIDGETS
# ============================================================
selector = widgets.Dropdown(
    options=items,
    description="Source:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "80px"}
)

run_btn = widgets.Button(
    description="▶  Process & Upscale",
    button_style="success",
    layout=widgets.Layout(width="220px", height="40px")
)

log_out = widgets.Output()


# ============================================================
# STEP 3 — MAIN PROCESSING CALLBACK
# ============================================================
def run_pipeline(button):
    with log_out:
        clear_output()

        selected_path = selector.value
        if not selected_path:
            print("❌ No selection made.")
            return

        source_basename = get_source_basename(selected_path)
        output_name = build_output_name(source_basename, output_format)

        global DRIVE_OUTPUT

        if DRIVE_MOUNTED:
            DRIVE_OUTPUT = os.path.join(
                "/content/drive/MyDrive/image upscaled",
                f"{source_basename}_upscaled_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{output_format}"
            )
            os.makedirs(DRIVE_OUTPUT, exist_ok=True)

        print("=" * 60)
        print(f"📂 Source  : {selected_path}")
        print(f"🏷️  Name    : {source_basename}")
        print(f"📁 Output  : {output_name}")
        print("=" * 60)

        # ── Clean & prepare staging folders ──────────────────
        for folder in (upload_folder, result_folder):
            if os.path.isdir(folder):
                shutil.rmtree(folder)
            os.makedirs(folder)

        # ── Copy / extract images into upload/ ───────────────
        try:
            if os.path.isdir(selected_path):
                print("📁 Folder detected — copying images…")
                count = copy_images_from_folder(selected_path, upload_folder)
                print(f"✅ Copied {count} images\n")
            elif selected_path.lower().endswith(".zip"):
                print("📦 ZIP detected — extracting images…")
                count = extract_images_from_zip(selected_path, upload_folder)
                print(f"✅ Extracted {count} images\n")
            else:
                print("❌ Unsupported source type.")
                return
        except Exception as e:
            print(f"❌ Error during copy/extract: {e}")
            return

        # ── List ready images ─────────────────────────────────
        input_images = sorted([
            f for f in os.listdir(upload_folder)
            if os.path.isfile(os.path.join(upload_folder, f))
        ])
        total = len(input_images)

        if total == 0:
            print("\033[91m❌ No images found in upload folder!\033[0m")
            return

        print(f"📝 Images to process: {total}")
        print(f"{'='*60}")
        print(f"🤖 Model         : {model_choice}")
        print(f"🎯 Upscale       : {outscale}x")
        print(f"🧩 Tile size     : {tile_size}px")
        print(f"✨ Face enhance  : {'Yes' if face_enhance else 'No'}")
        print(f"📄 Format        : {output_format.upper()}")
        if output_format in ("jpg", "jpeg", "webp"):
            print(f"🎨 Quality       : {jpg_quality}%")
        print(f"⚖️  Min size      : {min_file_size_mb} MB")
        print(f"📦 Max size      : {max_file_size_mb} MB" if max_file_size_mb > 0 else "📦 Max size      : Disabled")
        if DRIVE_MOUNTED:
            print(f"☁️  Drive backup  : {DRIVE_OUTPUT}")
        print(f"{'='*60}\n")

        # ── Upscale loop ──────────────────────────────────────
        batch_start = time.time()

        for idx, image_name in enumerate(input_images, 1):
            print(f"🔄 [{idx}/{total}] {image_name}")
            img_start  = time.time()
            input_path = os.path.join(upload_folder, image_name)
            base_name  = os.path.splitext(image_name)[0]

            # Run Real-ESRGAN
            cmd = (
                f'python inference_realesrgan.py '
                f'--model_path {selected_model} '
                f'-i "{input_path}" '
                f'-o {result_folder} '
                f'-s {outscale} '
                f'-t {tile_size} '
                f'--tile_pad {tile_pad} '
                f'--ext png '
                f"--suffix ''"
            )
            if face_enhance:
                cmd += " --face_enhance"
            get_ipython().system(cmd)

            time.sleep(0.5)

            # Detect newly created PNG
            existing_before = set(os.listdir(result_folder)) if idx > 1 else set()
            time.sleep(0.3)
            new_files = set(os.listdir(result_folder)) - existing_before
            temp_png  = (
                os.path.join(result_folder, list(new_files)[0])
                if new_files
                else os.path.join(result_folder, f"{base_name}.png")
            )

            output_file = f"{base_name}.{output_format}"
            output_path = os.path.join(result_folder, output_file)

            if not os.path.exists(temp_png):
                print(f"   ⚠️  Output not found — skipping\n")
                continue

            # Convert format
            img = Image.open(temp_png)
            save_with_quality(img, output_path, output_format, jpg_quality)

            # Remove temp PNG if we converted
            if temp_png != output_path and os.path.exists(temp_png):
                os.remove(temp_png)

            # Size enforcement
            enforce_min_size(output_path, output_format, jpg_quality, min_file_size_mb)
            enforce_max_size(output_path, output_format, jpg_quality, max_file_size_mb)

            # Stats
            dur = time.time() - img_start
            with Image.open(input_path) as im:
                iw, ih = im.size
            with Image.open(output_path) as om:
                ow, oh = om.size
            in_kb  = os.path.getsize(input_path) / 1024
            out_mb = os.path.getsize(output_path) / (1024 * 1024)

            print(f"   ⏱️  {dur:.2f}s")
            print(f"   📐 {iw}×{ih} ({iw*ih/1e6:.2f} MP) • {in_kb:.1f} KB  →  "
                  f"📏 {ow}×{oh} ({ow*oh/1e6:.2f} MP) • {out_mb:.2f} MB")

            # Real-time Drive backup
            if DRIVE_MOUNTED:
                drive_path = os.path.join(DRIVE_OUTPUT, output_file)
                shutil.copy2(output_path, drive_path)
                print(f"   ✅ Saved to Drive")

            print()

        elapsed = time.time() - batch_start

        # ── Rename results folder to output_name ─────────────
        named_folder = f"/content/{output_name}"
        if os.path.exists(named_folder):
            shutil.rmtree(named_folder)
        shutil.copytree(result_folder, named_folder)

        # ── Create ZIP ────────────────────────────────────────
        zip_filename = f"/content/{output_name}.zip"
        processed = [
            f for f in os.listdir(result_folder)
            if os.path.isfile(os.path.join(result_folder, f))
        ]
        processed_count = len(processed)

        print(f"{'='*60}")
        if processed_count == 0:
            print("❌ No results generated!")
            return

        print(f"📦 Creating ZIP: {output_name}.zip")
        get_ipython().system(f'zip -r -j -q "{zip_filename}" {result_folder}/*')

        # Upload ZIP to Drive
        if DRIVE_MOUNTED:
            zip_drive = os.path.join(DRIVE_OUTPUT, f"{output_name}.zip")
            shutil.copy2(zip_filename, zip_drive)
            print(f"☁️  ZIP uploaded to Drive: {zip_drive}")

        print(f"\n🎉 DONE!")
        print(f"✅ Processed     : {processed_count} images")
        print(f"⏱️  Total time    : {elapsed/60:.2f} min  ({elapsed/processed_count:.2f} sec/image)")
        print(f"🤖 Model         : {model_choice}")
        print(f"📁 Output folder : {named_folder}")
        print(f"📦 ZIP file      : {zip_filename}")
        if DRIVE_MOUNTED:
            print(f"☁️  Drive folder  : {DRIVE_OUTPUT}")
        print(f"{'='*60}\n")

        # ── Download / Link ───────────────────────────────────
        if auto_download_zip:
            from google.colab import files
            files.download(zip_filename)
            print("⬇️  ZIP download started automatically!")
        else:
            zip_basename = os.path.basename(zip_filename)
            folder_link  = named_folder.replace("/content/", "")

            display(HTML(f"""
<div style="
    font-family: sans-serif;
    background: transparent;
    color:#cdd6f4;
    padding:20px 24px;
    max-width:820px;
    margin-top:12px;
">
  <div style="font-size:18px;font-weight:700;margin-bottom:14px;">📦 Results Ready</div>

  <div style="margin-bottom:10px;">
    <span style="opacity:.7;font-size:13px;">🔹ZIP FILE</span><br>
    <code style="font-size:13px;">{zip_basename}</code>
  </div>

  <div style="margin-bottom:16px;">
    <span style="opacity:.7;font-size:13px;">🔹UPSCALED FOLDER</span><br>
    <code style="font-size:13px;">{output_name}/</code>
  </div>


  <span style="opacity:.8;font-size:12px;">
   🔹Browse <code style="font-size:13px;font-weight:700">"image upscaled/{folder_link}"</code> in your Google Drive.
  </span>
</div><br>
"""))

        print("\033[92m✅ All done!\033[0m")


# ============================================================
# RENDER UI
# ============================================================
run_btn.on_click(run_pipeline)

display(selector)
display(run_btn)
display(log_out)

▨▧▨▧▨▧▨▧▨▧▨▧▨▧▨▧▨▧▨▧▨▧

## **💥ফোল্ডার থেকে সমস্ত ডুপ্লিকেট ছবি মোছা💥**



In [ ]:
#@title 📌Remove Duplicate files
# ============================================================
# VISUAL DUPLICATE IMAGE REMOVER
# SELECT FOLDER OR ZIP FROM GOOGLE DRIVE
# ZIPS ARE EXTRACTED TO A REAL FOLDER
# DUPLICATES ARE MOVED (NOT DELETED)
# ============================================================

# Install dependencies
!pip -q install imagehash pillow tqdm ipywidgets

import os
import shutil
import zipfile
import ipywidgets as widgets

from PIL import Image
import imagehash
from tqdm import tqdm
from IPython.display import display, clear_output

# ============================================================
# CONFIG
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive"

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp",
    ".tiff",
    ".tif"
)

#@markdown ==================================================================
#@markdown ### 🔘 Duplicate Sensitivity Threshold Guide:
#@markdown - **1–2** = Very strict (almost exact duplicates only)
#@markdown - **3–5** = Safe (recommended)
#@markdown - **6–8** = Aggressive
#@markdown - **9–10** = Very aggressive
#@markdown
#@markdown Higher values detect more visually similar images but may increase false positives.
#@markdown
#@markdown ==================================================================
duplicate_sensitivity_threshold = 5 #@param {type:"slider", min:1, max:10, step:1}

print(
    f"Duplicate Sensitivity Threshold = "
    f"{duplicate_sensitivity_threshold}"
)

# ============================================================
# SCAN DRIVE
# ============================================================

print("🔍 Scanning Google Drive...")

items = []

for root, dirs, files in os.walk(DRIVE_ROOT):

    # Add folders
    for d in dirs:

        full_path = os.path.join(root, d)

        rel_path = os.path.relpath(
            full_path,
            DRIVE_ROOT
        )

        items.append(
            (
                f"📁 Folder | {rel_path}",
                full_path
            )
        )

    # Add zip files
    for f in files:

        if f.lower().endswith(".zip"):

            full_path = os.path.join(root, f)

            rel_path = os.path.relpath(
                full_path,
                DRIVE_ROOT
            )

            items.append(
                (
                    f"📦 ZIP | {rel_path}",
                    full_path
                )
            )

items.sort(key=lambda x: x[0].lower())

print(f"✅ Found {len(items)} folders/zip files")

# ============================================================
# UI
# ============================================================

selector = widgets.Dropdown(
    options=items,
    description="Source:",
    layout=widgets.Layout(width="95%"),
)

process_btn = widgets.Button(
    description="Find Duplicates",
    button_style="danger",
)

output = widgets.Output()

display(selector)
display(process_btn)
display(output)

# ============================================================
# FUNCTIONS
# ============================================================

def collect_images(folder):

    image_paths = []

    for root, _, files in os.walk(folder):

        # Skip duplicate_files folder
        if "duplicate_files" in root:
            continue

        for file in files:

            if file.lower().endswith(
                IMAGE_EXTENSIONS
            ):
                image_paths.append(
                    os.path.join(root, file)
                )

    return image_paths


def extract_zip_to_folder(zip_path):

    zip_name = os.path.splitext(
        os.path.basename(zip_path)
    )[0]

    extract_folder = os.path.join(
        os.path.dirname(zip_path),
        zip_name
    )

    os.makedirs(
        extract_folder,
        exist_ok=True
    )

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            extract_folder
        )

    return extract_folder


def find_duplicates(image_paths):

    hashes = {}
    duplicates = []

    print("\n🔍 Detecting duplicates...\n")

    for path in tqdm(image_paths):

        try:

            img = Image.open(path).convert(
                "RGB"
            )

            h = imagehash.phash(img)

            duplicate_found = False

            for existing_hash in hashes:

                if abs(
                    h - existing_hash
                ) <= duplicate_sensitivity_threshold:

                    duplicates.append(
                        (
                            path,
                            hashes[
                                existing_hash
                            ],
                        )
                    )

                    duplicate_found = True
                    break

            if not duplicate_found:

                hashes[h] = path

        except Exception as e:

            print(
                f"❌ Error processing: {path}"
            )

    return duplicates


def move_duplicates(
    duplicates,
    source_folder
):

    duplicate_folder = os.path.join(
        source_folder,
        "duplicate_files"
    )

    os.makedirs(
        duplicate_folder,
        exist_ok=True
    )

    moved = 0

    for dup_path, _ in duplicates:

        try:

            relative_path = os.path.relpath(
                dup_path,
                source_folder
            )

            destination = os.path.join(
                duplicate_folder,
                relative_path
            )

            os.makedirs(
                os.path.dirname(
                    destination
                ),
                exist_ok=True
            )

            shutil.move(
                dup_path,
                destination
            )

            moved += 1

        except Exception as e:

            print(
                f"❌ Failed to move: {dup_path}"
            )

    return moved


# ============================================================
# MAIN PROCESS
# ============================================================

def process_selection(btn):

    with output:

        clear_output()

        selected = selector.value

        if not selected:

            print(
                "❌ No source selected"
            )

            return

        print("=" * 60)
        print("Selected:")
        print(selected)
        print("=" * 60)

        # ----------------------------------------------------
        # FOLDER
        # ----------------------------------------------------

        if os.path.isdir(selected):

            source_folder = selected

            print(
                "\n📁 Folder selected"
            )

        # ----------------------------------------------------
        # ZIP
        # ----------------------------------------------------

        else:

            print(
                "\n📦 ZIP selected"
            )

            print(
                "📂 Extracting ZIP..."
            )

            source_folder = (
                extract_zip_to_folder(
                    selected
                )
            )

            print(
                f"✅ Extracted to:\n{source_folder}"
            )

        # ----------------------------------------------------
        # COLLECT IMAGES
        # ----------------------------------------------------

        image_paths = collect_images(
            source_folder
        )

        print(
            f"\n📷 Found {len(image_paths)} images"
        )

        if len(image_paths) == 0:

            print(
                "❌ No images found"
            )

            return

        # ----------------------------------------------------
        # FIND DUPLICATES
        # ----------------------------------------------------

        duplicates = find_duplicates(
            image_paths
        )

        print(
            f"\n🔍 Found {len(duplicates)} duplicates"
        )

        # ----------------------------------------------------
        # MOVE DUPLICATES
        # ----------------------------------------------------

        moved = move_duplicates(
            duplicates,
            source_folder
        )

        print(
            f"\n✅ Moved {moved} duplicate files"
        )

        print(
            "\n📂 Duplicate folder:"
        )

        print(
            os.path.join(
                source_folder,
                "duplicate_files"
            )
        )

        print(
            "\n🎉 Finished!"
        )

# ============================================================
# BUTTON EVENT
# ============================================================

process_btn.on_click(
    process_selection
)